# Random vs stratified vs balanced

How close each selection is to the month, and the weights of its games. Run `compare.py` first.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "config.yaml").exists():
    root = root.parent
sys.path.insert(0, str(root))

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

from src.config import CFG, COMPARISON_DIR
from src.common.buckets import SPEEDS, time_order
from src.common.plot_style import MONTH_COLOR, MUTED, PART_COLOR, SELECTION_COLOR, WIDTH, save

FIG_DIR = COMPARISON_DIR / "figures"
PARTS = ["train", "test"]
STEP_MIN = CFG["selection"]["window"] / 60

In [ ]:
buckets = pd.read_csv(COMPARISON_DIR / "buckets.csv", index_col="bucket")
by_time = pd.read_csv(COMPARISON_DIR / "by_time_left.csv")
weights = pd.read_csv(COMPARISON_DIR / "weights.csv")
result = json.loads((COMPARISON_DIR / "comparison.json").read_text(encoding="utf-8"))
names = result["selections"]
names

## Summary

chi2 per bucket near 1 means the selection is as close to the month as chance allows.

In [ ]:
rows = []
for name in names:
    for part in PARTS:
        s = result["stats"][name][part]
        rows.append({"selection": name, "part": part, "blunder probability (%)": 100 * s["blunder_rate"],
                     "chi2 per bucket": round(s["chi2_per_df"], 2), "abs(z) > 2 (%)": 100 * s["share_abs_z_over_2"],
                     "mean difference (points)": s["mean_abs_diff_pp"]})
for part in PARTS:
    d = result["fresh_draws"][part]
    rows.append({"selection": "fresh draws", "part": part,
                 "chi2 per bucket": f"{d['chi2_per_df_q05']:.2f}-{d['chi2_per_df_q95']:.2f}",
                 "mean difference (points)": d["mean_abs_diff_pp"]})
pd.DataFrame(rows).round(3)

## Blunder probability by time left

In [ ]:
fig, axes = plt.subplots(len(names), len(SPEEDS), figsize=(WIDTH, 1.9 * len(names)), sharey=True,
                         squeeze=False)
for row, name in zip(axes, names):
    for ax, speed in zip(row, SPEEDS):
        d = by_time[(by_time["speed"] == speed) & (by_time["moves"] > 0)]
        x = time_order(d["time_left"]).to_numpy() / 60
        ax.fill_between(x, 100 * d["rate_lo"], 100 * d["rate_hi"], color=MONTH_COLOR, alpha=.15, lw=0)
        ax.plot(x, 100 * d["rate"], color=MONTH_COLOR, lw=1.5)
        for i, part in enumerate(PARTS):
            col = f"{name}_{part}"
            rate = 100 * d[f"rate_{col}"].to_numpy()
            low = 100 * d[f"rate_lo_{col}"].to_numpy()
            high = 100 * d[f"rate_hi_{col}"].to_numpy()
            ax.errorbar(x + (i - .5) * .3 * STEP_MIN, rate, yerr=[rate - low, high - rate], fmt="o", ms=2.4,
                        lw=.7, capsize=0, color=PART_COLOR[part], mec="white", mew=.4)
        ax.set_title(f"{name}, {speed}")
        ax.set_xlim(left=0)
    row[0].set_ylabel("blunder probability (%)")
for ax in axes[-1]:
    ax.set_xlabel("time left (min)")
handles = [Line2D([], [], color=MONTH_COLOR, lw=1.5, label="month (95% interval)")]
for part in PARTS:
    handles.append(Line2D([], [], color=PART_COLOR[part], marker="o", ms=3.5, ls="", label=f"{part} part"))
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(.5, 1), ncol=3)
fig.tight_layout()
save(fig, FIG_DIR, "rate_by_time_left")
plt.show()

## How far each selection strays from the month

Left: z of every bucket of the training games (chance alone follows the dashed curve). Right: chi2 per bucket against fresh stratified draws.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(WIDTH, 2.4), gridspec_kw={"width_ratios": [1.6, 1]})
edges = np.linspace(-5, 5, 41)
for name in names:
    z = np.clip(buckets[f"z_{name}_train"].dropna().to_numpy(), -4.99, 4.99)
    a.hist(z, edges, density=True, histtype="step", lw=1.5, color=SELECTION_COLOR[name], label=name)
xs = np.linspace(-5, 5, 300)
a.plot(xs, np.exp(-xs ** 2 / 2) / np.sqrt(2 * np.pi), color=MONTH_COLOR, lw=1, ls="--", label="chance only")
a.set_xlabel("z of a bucket, training games")
a.set_ylabel("density")
a.legend(loc="upper left")

for i, part in enumerate(PARTS):
    draws = [d["chi2_per_df"] for d in result["fresh_draws"][part]["all"]]
    jitter = np.random.default_rng(i).uniform(-.12, .12, len(draws))
    b.scatter(i + jitter, draws, s=9, color=MUTED, alpha=.5, lw=0, label="fresh draws" if i == 0 else None)
    for name in names:
        b.scatter(i, result["stats"][name][part]["chi2_per_df"], marker="D", s=30, color=SELECTION_COLOR[name],
                  ec="white", lw=.6, zorder=3, label=name if i == 0 else None)
b.axhline(1, color=MUTED, lw=.8, ls=":")
b.set_xticks(range(len(PARTS)), PARTS)
b.set_xlim(-.6, len(PARTS) - .4)
b.set_ylabel("chi2 per bucket")
b.legend(loc="best")
fig.tight_layout(w_pad=1.5)
save(fig, FIG_DIR, "selection_z")
plt.show()

## Weights

Weight of a game = games of its group in the month / selected games of the group. Random gives every game the same weight; stratified and balanced give every group its own weight.

In [ ]:
elos = list(dict.fromkeys(weights["mean_elo"]))
width = .8 / len(names)
fig, axes = plt.subplots(1, len(SPEEDS), figsize=(WIDTH, 2.3), sharey=True)
for ax, speed in zip(axes, SPEEDS):
    for i, name in enumerate(names):
        d = weights[(weights["selection"] == name) & (weights["speed"] == speed)].set_index("mean_elo")
        d = d.reindex(elos)
        x = np.arange(len(elos)) + (i - (len(names) - 1) / 2) * width
        ax.bar(x, d["weight"], width, color=SELECTION_COLOR[name], label=name if speed == SPEEDS[0] else None)
    ax.set_title(speed)
    ax.set_xticks(range(len(elos)), elos, rotation=90)
    ax.set_xlabel("mean Elo of the game")
axes[0].set_ylabel("weight of a game")
fig.legend(loc="lower center", bbox_to_anchor=(.5, 1), ncol=len(names))
fig.tight_layout(w_pad=.6)
save(fig, FIG_DIR, "weights")
plt.show()